# Resultados finais (Fases 3 a 7)

Notebook da Fase 6. Reúne, numa ordem só, tudo o que vem depois dos treinos: predições, avaliação,
comparação entre experimentos, seeds, Grad-CAM, pacote do app e validação externa. A lógica está em
`src/chestxray/`; aqui só chamamos as funções e mostramos as tabelas e figuras que vão para a monografia.

Cada etapa confere se o que ela precisa existe e, se não existir, avisa e pula. Assim o notebook pode
rodar em partes:

| Etapa | Precisa de | Onde roda |
|---|---|---|
| 1. Predições (`preds_val.csv`, `preds_test.csv`) | `best.pt` e as imagens | PC com GPU (em CPU funciona, só demora) |
| 2–4. Avaliação, comparações, seeds, E5 | só os CSVs de predição | qualquer lugar, CPU |
| 5. Grad-CAM | `best.pt`, imagens e `BBox_List_2017.csv` | PC |
| 6. Pacote do app | `best.pt` e imagens de exemplo | PC |
| 7. CheXpert | `best.pt` e o CheXpert baixado | PC |

**Regra do plano (seção 10):** o modelo final é o E1, escolhido pela validação antes de qualquer teste.
Os demais experimentos são comparações (ablações) e não mudam essa escolha.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from chestxray import evaluate as ev
from chestxray.config import find_project_root, load_config

ROOT = find_project_root(Path.cwd())
PATHS_FILE = os.environ.get("CHESTXRAY_PATHS", "local.yaml")
PATHS = ROOT / "configs" / "paths" / PATHS_FILE
EXPERIMENTS = ROOT / "configs" / "experiments"

FINAL = "e1_baseline"                     # escolhido pela validação (plano, seção 10)
MAIN_SPLIT = ["e1_baseline", "e2_posweight", "e3_noaug", "e4_scratch"]
SEEDS = ["e1_baseline", "e1_baseline_seed43", "e1_baseline_seed44"]
OFFICIAL = ["e5_official"]                # outro conjunto de teste: tabela separada
PAIRS = [("e2_posweight", "e1_baseline"), ("e3_noaug", "e1_baseline"), ("e4_scratch", "e1_baseline")]
CHEXPERT_ROOT = os.environ.get("CHEXPERT_ROOT")  # ex.: E:/datasets/chexpert (etapa 7)

base = load_config(ROOT / "configs" / "base.yaml", PATHS)
RUNS = Path(base["paths"]["runs_dir"])
RESULTS = Path(base["paths"]["results_dir"])
CKPT = Path(base["paths"]["checkpoint_dir"])


def config_of(name):
    matches = [p for p in EXPERIMENTS.glob("*.yaml") if load_config(p)["experiment"] == name]
    return matches[0] if matches else None


def show_table(path):
    display(Markdown(Path(path).read_text(encoding="utf-8")))


def show_figure(path, width=700):
    if Path(path).exists():
        display(Image(filename=str(path), width=width))
    else:
        print("figura ainda não gerada:", path)


status = []
for name in dict.fromkeys(MAIN_SPLIT + SEEDS + OFFICIAL):
    run = RUNS / name
    status.append({"experimento": name,
                   "treinado": (run / "summary.json").exists(),
                   "best.pt": (CKPT / name / "best.pt").exists(),
                   "preds val/test": (run / "preds_val.csv").exists() and (run / "preds_test.csv").exists(),
                   "avaliado": (run / "metrics_test.json").exists()})
pd.DataFrame(status)

## 1. Predições

Gera `preds_val.csv` e `preds_test.csv` para cada experimento treinado que ainda não tem. Inferência em
fp32, com a mesma função do app e do Grad-CAM. É a única etapa que precisa das imagens (além das 5–7).

In [ ]:
from chestxray import inference

for row in status:
    name = row["experimento"]
    if row["preds val/test"]:
        continue
    if not row["best.pt"]:
        print(f"{name}: sem best.pt, pulando")
        continue
    print(f"{name}: gerando predições")
    inference.main(["--config", str(config_of(name)), "--paths", str(PATHS)])

## 2. Avaliação de cada experimento

Validação e teste de cada um, com IC95% por bootstrap de paciente. No teste, limiar e recalibração vêm
só da validação.

In [ ]:
for name in dict.fromkeys(MAIN_SPLIT + SEEDS + OFFICIAL):
    run = RUNS / name
    if not (run / "preds_test.csv").exists():
        print(f"{name}: sem predições, pulando")
        continue
    for split in ("val", "test"):
        if not (run / f"metrics_{split}.json").exists():
            ev.evaluate_run(run, split, RESULTS)

In [ ]:
show_table(RESULTS / "tables" / FINAL / "metricas_test.md")
show_figure(RESULTS / "figures" / FINAL / "roc_foco_test.png", 500)

### Comparação com a literatura (E1)

Comparação aproximada: o CheXNet usou outra divisão, outros hiperparâmetros e escolheu o modelo pela loss
de validação (plano, Fase 3).

In [ ]:
show_table(RESULTS / "tables" / FINAL / "comparacao_literatura.md")

### Limiares, calibração e subgrupos (E1)

In [ ]:
show_table(RESULTS / "tables" / FINAL / "limiares_teste.md")
show_table(RESULTS / "tables" / FINAL / "calibracao_teste.md")
show_figure(RESULTS / "figures" / FINAL / "calibracao_foco_test.png")
show_figure(RESULTS / "figures" / FINAL / "subgrupos_foco_test.png")

**Decisão para a Fase 5:** se, depois do Platt, as curvas de calibração das 3 classes ficarem perto da
diagonal, o app chama o valor de "probabilidade estimada" (`--label probability` na etapa 6) e a
decisão vai para a seção 10 do plano. Caso contrário, "escore do modelo" (padrão).

## 3. Experimentos lado a lado e diferenças pareadas

Todos no mesmo split (E1–E4). Diferença pareada = mesmas reamostras de paciente para os dois modelos;
se o IC95% contém zero, não houve diferença significativa.

In [ ]:
available = [n for n in MAIN_SPLIT if (RUNS / n / "metrics_test.json").exists()]
pairs = [f"{a}:{b}" for a, b in PAIRS if a in available and b in available]
# Diferenças pareadas só no teste: a escolha pela validação (E1 × E2) já está registrada no plano
ev.main(["--compare", *[str(RUNS / n) for n in available], "--split", "val"])
ev.main(["--compare", *[str(RUNS / n) for n in available], "--split", "test", "--pairs", *pairs])
show_table(RESULTS / "tables" / "experimentos_test.md")
show_table(RESULTS / "tables" / "diferencas_pareadas_test.md")

## 4. Seeds e split oficial

As 3 seeds mostram quanto a AUC varia só pela aleatoriedade do treino. Uma diferença entre experimentos
menor que essa variação não deve ser interpretada. O E5 usa o split oficial do NIH, cujo teste é mais
"doente": comparar a AUPRC sempre com a prevalência ao lado.

In [ ]:
done = [n for n in SEEDS if (RUNS / n / "metrics_test.json").exists()]
if len(done) >= 2:
    ev.main(["--seeds", *[str(RUNS / n) for n in done]])
    show_table(RESULTS / "tables" / "seeds_test.md")
else:
    print("seeds ainda não avaliadas:", [n for n in SEEDS if n not in done])

for name in OFFICIAL:
    if (RUNS / name / "metrics_test.json").exists():
        show_table(RESULTS / "tables" / name / "metricas_test.md")
        show_table(RESULTS / "tables" / name / "comparacao_literatura.md")

## 5. Grad-CAM (Fase 4)

Galeria com critério fixo (VP e FP de maior escore, FN de menor escore, uma imagem por paciente),
comparação das duas camadas-alvo, caixas do radiologista e pointing game.

In [ ]:
from chestxray import gradcam

if (CKPT / FINAL / "best.pt").exists() and (RUNS / FINAL / "calibration.json").exists():
    gradcam.main(["--config", str(config_of(FINAL)), "--paths", str(PATHS)])
    for name in ("galeria_pneumonia", "galeria_atelectasis", "galeria_effusion", "camadas_foco"):
        show_figure(RESULTS / "figures" / FINAL / "gradcam" / f"{name}.png", 650)
    show_table(RESULTS / "tables" / FINAL / "pointing_game.md")
else:
    print("precisa do best.pt e da avaliação de teste do", FINAL)

## 6. Pacote do app (Fase 5)

Exporta `app/model/` (pesos, calibração e exemplos) e confere que o app reproduz o `preds_test.csv`.
Depois: `python app/app.py` para testar no navegador, e o README explica como publicar no Hugging Face.

In [ ]:
from chestxray import demo

LABEL = "score"  # "probability" só se a calibração da etapa 2 justificar (ver a decisão acima)
if (CKPT / FINAL / "best.pt").exists():
    demo.main(["--config", str(config_of(FINAL)), "--paths", str(PATHS), "--out", str(ROOT / "app" / "model"),
               "--label", LABEL])

## 7. Validação externa no CheXpert (Fase 7, complementar)

Defina `CHEXPERT_ROOT` com a pasta do CheXpert (a que contém `valid.csv`, ou a pasta acima dela) antes
de abrir o Jupyter.

In [ ]:
from chestxray import external

if CHEXPERT_ROOT and (CKPT / FINAL / "best.pt").exists():
    external.main(["--config", str(config_of(FINAL)), "--paths", str(PATHS), "--chexpert-root", CHEXPERT_ROOT])
    show_table(RESULTS / "tables" / FINAL / "validacao_externa_chexpert.md")
    show_figure(RESULTS / "figures" / FINAL / "roc_foco_chexpert.png", 500)
else:
    print("defina CHEXPERT_ROOT para rodar esta etapa")